# 01 · Prompt Chaining with LangChain

Prompt chaining is the simplest — and most fundamental — pattern in LangChain. You connect a sequence of LLM calls where **the output of one step becomes the input of the next**.

This notebook builds a **3-step blog post pipeline**:

```
topic  →  [Step 1: Research angle]  →  [Step 2: Outline]  →  [Step 3: Intro paragraph]
```

---

## Setup

In [2]:
# Install dependencies
%pip install -qU langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv

load_dotenv("../.env")

# Model comes from .env in "provider:model" form (e.g. openai:gpt-4o-mini),
# so switching models is a config change, not a code change.
llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0.7)

---

## Step 1 — Define each prompt in the chain

Each step is a `ChatPromptTemplate` + `llm` + `StrOutputParser`.  
The output parser converts the LLM's `AIMessage` response into a plain string so it can be passed to the next step cleanly.

In [4]:
# Step 1: Find a compelling research angle for the topic
angle_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a tech content strategist. Your job is to find a sharp, "
               "non-obvious angle for a developer-focused article."),
    ("human", "Topic: {topic}\n\nSuggest ONE compelling research angle in 2-3 sentences. "
              "Be specific. Avoid generic takes.")
])

# Step 2: Generate a structured outline based on the angle
outline_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Create clear, developer-friendly article outlines."),
    ("human", "Research angle: {angle}\n\nWrite a 5-section article outline with a title "
              "and one-line description for each section.")
])

# Step 3: Write an intro paragraph based on the outline
intro_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a developer advocate who writes engaging technical content. "
               "Your intros hook the reader immediately."),
    ("human", "Article outline:\n{outline}\n\nWrite a compelling intro paragraph (4-5 sentences) "
              "that hooks a developer audience.")
])

parser = StrOutputParser()

---

## Step 2 — Build the chain with LCEL

LCEL (LangChain Expression Language) uses the `|` pipe operator to connect steps.  
Each step passes its string output to the next prompt via a key in the template.

```
angle_chain:   topic  →  angle_prompt  →  llm  →  parser  →  angle (str)
outline_chain: angle  →  outline_prompt →  llm  →  parser  →  outline (str)
intro_chain:   outline →  intro_prompt  →  llm  →  parser  →  intro (str)
```

In [5]:
from langchain_core.runnables import RunnablePassthrough

# Build each sub-chain
angle_chain   = angle_prompt   | llm | parser
outline_chain = outline_prompt | llm | parser
intro_chain   = intro_prompt   | llm | parser

# Connect them: pass outputs forward using RunnablePassthrough
full_chain = (
    {"angle": angle_chain}                          # Step 1: topic → angle
    | RunnablePassthrough.assign(                   # Step 2: angle → outline (keep angle too)
        outline=outline_chain
    )
    | RunnablePassthrough.assign(                   # Step 3: outline → intro (keep all)
        intro=intro_chain
    )
)

> **What is `RunnablePassthrough.assign`?**  
> It runs a new chain and adds its output as a new key to the existing dict — without losing the previous keys.  
> This is how you "accumulate" outputs across steps in a chain.

---

## Step 3 — Run the chain

In [6]:
result = full_chain.invoke({"topic": "LangChain prompt chaining"})

print("=" * 60)
print("ANGLE")
print("=" * 60)
print(result["angle"])

print("\n" + "=" * 60)
print("OUTLINE")
print("=" * 60)
print(result["outline"])

print("\n" + "=" * 60)
print("INTRO PARAGRAPH")
print("=" * 60)
print(result["intro"])

ANGLE
Explore how prompt chaining in LangChain can be optimized through adaptive context management, specifically investigating techniques for dynamically pruning or reorganizing intermediate prompts based on real-time model feedback. This angle uncovers potential performance improvements and accuracy gains in multi-step reasoning tasks, moving beyond static prompt sequences to a more intelligent, resource-aware chaining strategy.

OUTLINE
**Title:**  
Enhancing Prompt Chaining in LangChain with Adaptive Context Management for Dynamic Reasoning

**Outline:**

1. **Introduction: The Need for Intelligent Prompt Chaining in Language Models**  
   Overview of prompt chaining in LangChain, challenges of static sequences, and the motivation for dynamic, resource-aware strategies.

2. **Fundamentals of Adaptive Context Management in Multi-Step Reasoning**  
   Explanation of context management concepts, including the importance of relevancy, model feedback, and the limitations of fixed prompt

---

## Bonus — Stream the final output

Instead of waiting for the full chain to complete, stream the last step token by token.  
This is useful in any user-facing app.

In [7]:
# Run the first two steps normally to get the outline
intermediate = (
    {"angle": angle_chain}
    | RunnablePassthrough.assign(outline=outline_chain)
).invoke({"topic": "LangChain prompt chaining"})

# Stream the final step
print("Streaming intro...\n")
for chunk in intro_chain.stream({"outline": intermediate["outline"]}):
    print(chunk, end="", flush=True)

Streaming intro...

Imagine a debugging session where your AI not only identifies issues but actively learns from each step, correcting its own mistakes and adapting to new information—turning a single prompt into a multi-turn, self-improving conversation. With prompt chaining in LangChain, this vision becomes a reality, transforming static interactions into dynamic, context-aware workflows that elevate your debugging toolkit. By leveraging multi-step reasoning and iterative feedback, you can build AI systems that are smarter, more reliable, and less prone to hallucinations. Ready to unlock the full potential of your LLMs and take your debugging workflows to the next level? Let's dive in.

---

## What to remember

| Concept | What it does |
|---|---|
| `\|` pipe operator | Connects runnables — output of left becomes input of right |
| `StrOutputParser` | Converts `AIMessage` → plain `str` for the next step |
| `RunnablePassthrough` | Passes the current input through unchanged |
| `RunnablePassthrough.assign(key=chain)` | Runs a chain, adds result as a new key to the dict |
| `.stream()` | Yields output tokens one by one instead of waiting for the full response |

---

## What's next

In **Article 02 — Parallelization**, we'll run multiple chains at the same time with `RunnableParallel` instead of waiting for each step to finish sequentially.